In [7]:
# If needed, install once:
# !pip install tensorflow adversarial-robustness-toolbox scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import ProjectedGradientDescentTensorFlowV2
from art.defences.trainer import AdversarialTrainer

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [8]:
# Configuration
def normalize_path(path_like) -> Path:
    return Path(str(path_like).replace("\\", "/")).expanduser()


def resolve_existing_path(*candidates, required: bool = True) -> Path:
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        p = normalize_path(candidate)
        variants = [p]
        if not p.is_absolute():
            variants.extend([
                Path.cwd() / p,
                Path.cwd().parent / p,
                Path.cwd().parent.parent / p,
            ])
        for variant in variants:
            checked.append(str(variant))
            if variant.exists():
                print(f"Resolved path: {variant}")
                return variant
    if required:
        raise FileNotFoundError(f"Could not find path in {checked}")
    return None


CDATA_DIR = resolve_existing_path(
    "./data",
    "./datasets",
    "../data",
    "../datasets",
    required=False,
)

ARTIFACT_DIR = Path("./artifacts").resolve()
RESULTS_DIR = Path("./results").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Data loading
LABEL_COL = "anomaly"
EXTRA_DROP = "channel"
DROP_COLS = [LABEL_COL, EXTRA_DROP]
TEST_SIZE = 0.2

# Model hyperparameters
LR = 0.001
L2_REG = 0.0001
BATCH_SIZE = 32
NB_EPOCHS = 100

# PGD attack parameters
PGD_EPSILON = 0.3
PGD_EPS_STEP = 0.1
PGD_MAX_ITER = 40
PGD_BATCH_SIZE = 32

# Adversarial training
ADV_RATIO = 0.5

# Model saving
SAVE_MODELS = True

print("Configuration set.")

Configuration set.


In [9]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()
    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )
    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    return X, y, feature_cols


def make_balanced_training_set(X, y, seed=SEED):
    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError(f"Expected binary labels, got: {classes}")

    np.random.seed(seed)
    indices_0 = np.where(y == classes[0])[0]
    indices_1 = np.where(y == classes[1])[0]

    if len(indices_0) < len(indices_1):
        indices_0 = np.random.choice(indices_0, size=len(indices_1), replace=True)
    else:
        indices_1 = np.random.choice(indices_1, size=len(indices_0), replace=True)

    indices = np.concatenate([indices_0, indices_1])
    np.random.shuffle(indices)

    return X[indices], y[indices]


def to_one_hot(y, num_classes):
    one_hot = np.zeros((len(y), num_classes), dtype=np.float32)
    one_hot[np.arange(len(y)), y] = 1.0
    return one_hot


print("Utility functions defined.")

Utility functions defined.


In [10]:
def build_logistic_model(d_in: int, l2_reg: float = L2_REG):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(d_in,)),
        tf.keras.layers.Dense(
            2,
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name="logistic_output",
        ),
    ])


def make_art_classifier(d_in: int, lr: float = LR, l2_reg: float = L2_REG):
    model = build_logistic_model(d_in=d_in, l2_reg=l2_reg)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    loss_object = tf.keras.losses.CategoricalCrossentropy()

    @tf.function
    def train_step(model_instance, x_batch, y_batch):
        with tf.GradientTape() as tape:
            predictions = model_instance(x_batch, training=True)
            loss = loss_object(y_batch, predictions)
        gradients = tape.gradient(loss, model_instance.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_instance.trainable_variables))
        return loss

    model._train_step = train_step
    return TensorFlowV2Classifier(
        model=model,
        loss_object=loss_object,
        train_step=train_step,
        nb_classes=2,
        input_shape=(d_in,)
    )


def eval_classifier(art_clf, X, y, description: str):
    y_pred_probs = art_clf.predict(X).astype(np.float32)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, zero_division=0)

    print(f"{description}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print()

    return {
        "description": description,
        "accuracy": acc,
        "f1_score": f1,
    }, y_pred


print("Model building functions defined.")

Model building functions defined.


In [11]:
# Load dataset
csv_file = resolve_existing_path(
    Path("../../CSVs/dataset.csv"),
    Path("../CSVs/dataset.csv"),
    Path("CSVs/dataset.csv"),
)

X, y, feature_cols = load_and_prepare(str(csv_file))
print(f"Loaded data: X.shape={X.shape}, y.shape={y.shape}")
print(f"Label distribution: {np.bincount(y)}")

# Normalize
scaler = MinMaxScaler()
X = scaler.fit_transform(X).astype(np.float32)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Balance training set
X_train_balanced, y_train_balanced = make_balanced_training_set(X_train, y_train)
y_train_balanced_oh = to_one_hot(y_train_balanced, 2)

print(f"X_train_balanced: {X_train_balanced.shape}, y: {y_train_balanced.shape}")
print(f"X_test: {X_test.shape}, y: {y_test.shape}")

Resolved path: ..\..\CSVs\dataset.csv
Loaded data: X.shape=(2123, 21), y.shape=(2123,)
Label distribution: [1689  434]
X_train_balanced: (2702, 21), y: (2702,)
X_test: (425, 21), y: (425,)


In [12]:
# Step 1: Train clean logistic regression model
print("Training clean TensorFlow v2 logistic regression with:", {
    "learning_rate": LR,
    "l2_reg": L2_REG,
    "batch_size": BATCH_SIZE,
    "epochs": NB_EPOCHS,
    "pgd_epsilon": PGD_EPSILON,
    "pgd_eps_step": PGD_EPS_STEP,
    "pgd_max_iter": PGD_MAX_ITER,
    "adv_ratio": ADV_RATIO,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train_balanced, y_train_balanced_oh, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    art_clean.model.save_weights(str(ARTIFACT_DIR / "logreg_tf2_clean.weights.h5"))
    print(f"Saved: {ARTIFACT_DIR / 'logreg_tf2_clean.weights.h5'}")

Training clean TensorFlow v2 logistic regression with: {'learning_rate': 0.001, 'l2_reg': 0.0001, 'batch_size': 32, 'epochs': 100, 'pgd_epsilon': 0.3, 'pgd_eps_step': 0.1, 'pgd_max_iter': 40, 'adv_ratio': 0.5}
clean_model_on_clean_test:
  Accuracy: 0.9106
  F1-Score: 0.7865

Saved: C:\Users\Jan\Git\AnomalyDetection\CONSGATE Notebooks\PGD\artifacts\logreg_tf2_clean.weights.h5


In [13]:
# Step 2: initialize PGD and generate adversarial samples
def make_pgd_attack(classifier):
    return ProjectedGradientDescentTensorFlowV2(
        estimator=classifier,
        eps=PGD_EPSILON,
        eps_step=PGD_EPS_STEP,
        max_iter=PGD_MAX_ITER,
        batch_size=PGD_BATCH_SIZE,
        targeted=False,
        verbose=False,
    )

pgd_attack = make_pgd_attack(art_clean)

X_test_adv = np.clip(pgd_attack.generate(x=X_test), 0.0, 1.0).astype(np.float32)
X_train_adv = np.clip(pgd_attack.generate(x=X_train), 0.0, 1.0).astype(np.float32)

y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

X_test_combined = np.vstack([X_test, X_test_adv])
y_test_combined = np.concatenate([y_test, y_test_adv])

print(f"X_train_adv: {X_train_adv.shape}")
print(f"X_test_adv: {X_test_adv.shape}")
print(f"X_test_combined: {X_test_combined.shape}")

X_train_adv: (1698, 21)
X_test_adv: (425, 21)
X_test_combined: (850, 21)


In [14]:
# Evaluate performance of the clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)

clean_model_on_adv_test:
  Accuracy: 0.0918
  F1-Score: 0.0853

clean_model_on_combined_test:
  Accuracy: 0.5012
  F1-Score: 0.2933



In [15]:
# Step 3: Retrain using ART's AdversarialTrainer with the PGD attack
art_adv = make_art_classifier(d_in=X_train.shape[1])
pgd_for_training = make_pgd_attack(art_adv)

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=pgd_for_training,
    ratio=ADV_RATIO,
)

adv_trainer.fit(
    X_train_balanced,
    y_train_balanced_oh,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    art_adv.model.save_weights(str(ARTIFACT_DIR / "logreg_tf2_adversarial_trained.weights.h5"))
    print(f"Saved: {ARTIFACT_DIR / 'logreg_tf2_adversarial_trained.weights.h5'}")

Adversarial training epochs: 100%|██████████| 100/100 [1:17:36<00:00, 46.56s/it]  

Saved: C:\Users\Jan\Git\AnomalyDetection\CONSGATE Notebooks\PGD\artifacts\logreg_tf2_adversarial_trained.weights.h5


In [16]:
# Step 4: Test using X_test_adv and the combined clean+adv test set
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

# Also check whether adversarial training preserved clean performance
adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)

adv_trained_model_on_adv_test:
  Accuracy: 0.4941
  F1-Score: 0.3344

adv_trained_model_on_clean_test:
  Accuracy: 0.6965
  F1-Score: 0.4464

adv_trained_model_on_combined_test:
  Accuracy: 0.5953
  F1-Score: 0.3813



In [17]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

display(summary_df)

summary_path = RESULTS_DIR / "logreg_tf2_pgd_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")

,description,accuracy,f1_score
0,clean_model_on_clean_test,0.910588,0.786517
1,clean_model_on_adv_test,0.091765,0.085308
2,clean_model_on_combined_test,0.501176,0.293333
3,adv_trained_model_on_clean_test,0.696471,0.446352
4,adv_trained_model_on_adv_test,0.494118,0.334365
5,adv_trained_model_on_combined_test,0.595294,0.381295


Saved: C:\Users\Jan\Git\AnomalyDetection\CONSGATE Notebooks\PGD\Results\logreg_tf2_pgd_pipeline_summary.csv


In [18]:
# Dual-stream consistency gate

CONFIDENCE_THRESHOLD = 0.60
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(art_clf, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    preds = np.argmax(probs, axis=1)
    return preds, probs


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        reasons = []
        confidences_nominal = np.max(probs_nominal, axis=1)

        for i in range(n_samples):
            disagreement = int(preds_nominal[i] != preds_guardian[i])
            low_confidence = int(confidences_nominal[i] < confidence_threshold)

            if disagreement and low_confidence:
                flags[i] = 1
                reasons.append("disagreement_and_low_confidence")
            else:
                reasons.append("clean")

        return flags, reasons, preds_nominal, probs_nominal, preds_guardian, probs_guardian


detector = DualStreamDetector(art_clean, art_adv)
print("DualStreamDetector initialized.")

DualStreamDetector initialized.


In [19]:
# Apply dual-stream detection on all test sets
flags_clean, reasons_clean, preds_nom_clean, probs_nom_clean, preds_grd_clean, probs_grd_clean = detector.detect_attacks(X_test)
flags_adv, reasons_adv, preds_nom_adv, probs_nom_adv, preds_grd_adv, probs_grd_adv = detector.detect_attacks(X_test_adv)
flags_combined, reasons_combined, preds_nom_combined, probs_nom_combined, preds_grd_combined, probs_grd_combined = detector.detect_attacks(X_test_combined)

dual_stream_prediction_summary_df = pd.DataFrame({
    "test_set": ["clean", "adv", "combined"],
    "total_samples": [len(X_test), len(X_test_adv), len(X_test_combined)],
    "flagged_as_attack": [
        np.sum(flags_clean),
        np.sum(flags_adv),
        np.sum(flags_combined),
    ],
    "pct_flagged": [
        100.0 * np.sum(flags_clean) / len(X_test),
        100.0 * np.sum(flags_adv) / len(X_test_adv),
        100.0 * np.sum(flags_combined) / len(X_test_combined),
    ],
})

display(dual_stream_prediction_summary_df)

gate_clean_df = pd.DataFrame({
    "index": np.arange(len(X_test)),
    "ground_truth": y_test,
    "nominal_pred": preds_nom_clean,
    "nominal_conf": np.max(probs_nom_clean, axis=1),
    "guardian_pred": preds_grd_clean,
    "guardian_conf": np.max(probs_grd_clean, axis=1),
    "disagreement": (preds_nom_clean != preds_grd_clean).astype(int),
    "flagged": flags_clean,
    "reason": reasons_clean,
})

gate_adv_df = pd.DataFrame({
    "index": np.arange(len(X_test_adv)),
    "ground_truth": y_test_adv,
    "nominal_pred": preds_nom_adv,
    "nominal_conf": np.max(probs_nom_adv, axis=1),
    "guardian_pred": preds_grd_adv,
    "guardian_conf": np.max(probs_grd_adv, axis=1),
    "disagreement": (preds_nom_adv != preds_grd_adv).astype(int),
    "flagged": flags_adv,
    "reason": reasons_adv,
})

gate_combined_df = pd.DataFrame({
    "index": np.arange(len(X_test_combined)),
    "ground_truth": y_test_combined,
    "nominal_pred": preds_nom_combined,
    "nominal_conf": np.max(probs_nom_combined, axis=1),
    "guardian_pred": preds_grd_combined,
    "guardian_conf": np.max(probs_grd_combined, axis=1),
    "disagreement": (preds_nom_combined != preds_grd_combined).astype(int),
    "flagged": flags_combined,
    "reason": reasons_combined,
})

dual_stream_detection_results = pd.DataFrame({
    "metric": [
        "clean_test_false_positive_rate",
        "adv_test_true_positive_rate",
        "combined_test_flag_rate",
    ],
    "value": [
        np.sum(flags_clean) / len(X_test) if len(X_test) > 0 else 0,
        np.sum(flags_adv) / len(X_test_adv) if len(X_test_adv) > 0 else 0,
        np.sum(flags_combined) / len(X_test_combined) if len(X_test_combined) > 0 else 0,
    ],
})

display(dual_stream_detection_results)

,test_set,total_samples,flagged_as_attack,pct_flagged
0,clean,425,13,3.058824
1,adv,425,0,0.000000
2,combined,850,13,1.529412


,metric,value
0,clean_test_false_positive_rate,0.030588
1,adv_test_true_positive_rate,0.000000
2,combined_test_flag_rate,0.015294


In [20]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "logreg_tf2_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "logreg_tf2_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")

Saved: C:\Users\Jan\Git\AnomalyDetection\CONSGATE Notebooks\PGD\Results\logreg_tf2_dual_stream_prediction_summary.csv
Saved: C:\Users\Jan\Git\AnomalyDetection\CONSGATE Notebooks\PGD\Results\logreg_tf2_dual_stream_detection_results.csv
